<a href="https://colab.research.google.com/github/lexoz-bedra/rank_model_vk/blob/main/vk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import ndcg_score
from xgboost import XGBClassifier, XGBRanker

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from google.colab import drive

In [ ]:
drive.mount('/content/drive/')

In [ ]:
data = pd.read_csv('/content/drive/My Drive/vk/intern_task_1.csv')

In [ ]:
data.head(3)

# EDA & preprocessing

In [ ]:
only_vals = []

for col in data.drop(columns=['query_id', 'rank']).columns:
  if data[col].nunique() == 1:
    print(col, data[col].unique())
    only_vals.append(col)

In [ ]:
data = data.drop(columns=only_vals)

In [ ]:
print(f'Shape: {data.shape}')
print(f'Columns: {data.columns}')

In [ ]:
print(data.shape)
data.drop_duplicates(inplace=True)
print(data.shape)

In [ ]:
data.isnull().sum().unique()

In [ ]:
print(data.shape)
data.dropna(inplace=True)
print(data.shape)

In [ ]:
data.dtypes.unique()

In [ ]:
data['rank'].unique()

In [ ]:
counts = data['query_id'].value_counts()

counts

In [ ]:
corr = data.corr(method='pearson')

In [ ]:
for col in corr.columns:
  for row in corr.iterrows():
    if corr[row[0]][col] > 0.9 and corr[row[0]][col] < 1:
      print(row[0], col, corr[row[0]][col])

In [ ]:
corr['rank'].drop('rank').sort_values(ascending=False)

In [ ]:
corr['rank']['query_id']

?? ???????, ??? ? ???????? ??? ???????? ?????????, ????????? ???, ?????????? ???, ??????? ?????????? ? ?????? ?? ? ?????? ??????????? ???????? ???. ?????? ???????????? ???????????????????? (????????? ???? ????? ?????? ??????????? ???? ? ??????), ??????? ? ???????? ?????? ????? ???????????? L1-?????????????.

????? ????? ????????, ??? ???? ?? ???????? ??? ?????? `query_id`, ?? ???? ? ?????? ????? ?????? ?? ????? ???????? ????????? ????????? ??????????? ???????????.
? ?????? ?????? ?????????? ????? ???????????? pointwise ??? pairwise ?????? ??? ???????? ????????????. ????????? ?????? ?????? ? ?????, ????? ?????, ?? ????????? ???????????.

Train ? test-???????? ??????? ?? ???????? ??????? ??????? `train_test_split` ?? sklearn. ????? ??????? ????????????? ???????, ??? ??? ? ??? ???????? ????? ??????.

# Models development & training

???????? ? ?????? 2 ??????: XGBClassifier ? XGBRanker. ?????????, ??? ????????? ????? ??? ?????? ????????????.

? ???????? ?????? ????? ???????????? nDCG@5 ? pFound.

In [ ]:
X = data.drop(['query_id', 'rank'], axis=1)
y = data['rank']
query_id = data['query_id']

In [ ]:
X_train, X_temp, y_train, y_temp, query_id_train, query_id_temp = train_test_split(
    X, y, query_id, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test, query_id_val, query_id_test = train_test_split(
    X_temp, y_temp, query_id_temp, test_size=0.5, random_state=42)

In [ ]:
eval_set = [(X_val, y_val)]

## XGBoost Classifier

In [ ]:
xgb_clf = XGBClassifier(n_estimators=1000, learning_rate=0.1,
                        max_depth=6,
                        objective='multi:softprob',
                        verbosity=0, use_label_encoder=False,
                        tree_method='hist', reg_alpha=0.005,
                        eval_set=eval_set, verbose=True, device='gpu')

In [ ]:
xgb_clf.fit(X_train, y_train, verbose=True)

In [ ]:
# ??????? ndcg@5

y_pred_xgb_clf = xgb_clf.predict(X_test)

def avg_ndcg(y_true, y_score, query_id_score, cnt):
    ndcg_scores = []
    for qid in np.unique(query_id_score):
        indices = np.where(query_id_score == qid)[0]
        true_rel = y_true.iloc[indices].values
        pred_scores = y_score[indices]

        if len(true_rel) == 1:
            continue

        rel_sorted_idx = np.argsort(-pred_scores, axis=0)[:]
        sorted_pred_scores = pred_scores[rel_sorted_idx]

        k = min(cnt, len(true_rel))
        ndcg = ndcg_score([true_rel], [sorted_pred_scores], k=k)
        ndcg_scores.append(ndcg)

    average_ndcg = np.mean(ndcg_scores) if ndcg_scores else float('nan')

    return average_ndcg

In [ ]:
print(f'Average NDCG@5: {avg_ndcg(y_test, y_pred_xgb_clf, query_id_test, 5)}')

In [ ]:
# ??????? pFound

def pfound(y_true, y_score, query_ids, p_break=0.15):

    y_true = y_true.values

    p_found_values = []

    for qid in np.unique(query_ids):
        indices = np.where(query_ids == qid)[0]

        y_true_normalized = y_true[indices] / 4
        y_score_sorted = y_score[indices]
        order = np.argsort(-y_score_sorted)
        y_true_sorted = y_true_normalized[order]

        p_look = 1
        p_found = 0

        for rel in y_true_sorted:
            p_found += p_look * rel
            p_look *= (1 - rel) * (1 - p_break)

        p_found_values.append(p_found)

    return np.mean(p_found_values)

In [ ]:
print(f'PFound: {pfound(y_test, y_pred_xgb_clf, query_id_test)}')

## XGBoost Ranker

In [ ]:
xgb_rank = XGBRanker(n_estimators=1000, learning_rate=0.1,
                        max_depth=6,
                        objective='rank:pairwise',
                        verbosity=0, use_label_encoder=False,
                        tree_method='hist', reg_alpha=0.005, verbose=True,
                        eval_set=eval_set, device='gpu')

In [ ]:
sorted_idx = query_id_train.argsort()
X_train_sorted = X_train.iloc[sorted_idx]
y_train_sorted = y_train.iloc[sorted_idx]
query_id_train_sorted = query_id_train.iloc[sorted_idx]

group_train = query_id_train_sorted.groupby(query_id_train_sorted).size().values

In [ ]:
xgb_rank.fit(X_train_sorted, y_train_sorted, group=group_train)

In [ ]:
y_pred_xgb_rank = xgb_rank.predict(X_test)

In [ ]:
print(f'Average NDCG@5: {avg_ndcg(y_test, y_pred_xgb_rank, query_id_test, 5)}')

In [ ]:
print(f'PFound: {pfound(y_test, y_pred_xgb_rank, query_id_test)}')

?????? ???????? NDCG@5 ? ????? ??????? ????? ????????????????? ? ???, ??? ? ??????? ??????? ?????????? ??????????? ???????????, ? ????? ? ???, ??? ?????? ?????? ??? ?????????????? ????? ???????? ??? ???? ?????? ??? ????? ????????.

???????? pFound - ???????????, ??? ? ?????? ?????? ???????????? ?????? ???-?? ?????????? ???????????, ?????? ??? ????????? ???? ????? - ???????? ??????? ? ????? ??????? ? ??????? ???? ?? ??????, ??? ????? ????????????????? ? ???, ??? ?????? ?????? ?????? ?????? ???????????? ? ??????? ????????? ???? ?????.